# Contoso Private Banking — Agent Queries (intent-level demo)

Interactive queries against `aria-rm-briefing-agent` (deployed in
[`08-05b-01-private-banking-agent-setup.ipynb`](08-05b-01-private-banking-agent-setup.ipynb)).

The first half exercises the headline workflow (one client briefing in one tool
call). The second half makes the **intent-level vs endpoint-level contrast
explicit**: the same workflow, simulated as if the agent only had endpoint-style
tools — which forces 5+ chained calls, more tokens, and more opportunities for
the planner to pick wrong.


## Prerequisites

- 08-05b-01 has been run successfully (the agent exists on the admin project).

## Imports and configuration

In [1]:
import hashlib
import os
import subprocess
from pathlib import Path

from dotenv import load_dotenv

repo_root = Path(
    subprocess.run('git rev-parse --show-toplevel', shell=True, capture_output=True, text=True).stdout.strip()
)
load_dotenv(repo_root / '.env', override=True)

CHAT_MODEL = os.environ.get('CHAT_MODEL', 'gpt-4.1-mini')

SUBSCRIPTION_ID = (
    os.environ.get('AZURE_SUBSCRIPTION_ID')
    or subprocess.run('az account show --query id -o tsv',
                      shell=True, capture_output=True, text=True).stdout.strip()
)
SUFFIX           = hashlib.sha256((SUBSCRIPTION_ID + 'v2').encode()).hexdigest()[:6]
PROJECT_ENDPOINT = f'https://aif-core-{SUFFIX}.services.ai.azure.com/api/projects/project-admin-{SUFFIX}'

PB_MCP_RG     = os.environ.get('PRIVATE_BANKING_MCP_RESOURCE_GROUP', 'rg-foundry-private-banking-mcp')
_mcp_suffix   = hashlib.md5(f'{SUBSCRIPTION_ID}-{PB_MCP_RG}'.encode()).hexdigest()[:6]
FUNC_APP_NAME = os.environ.get('PRIVATE_BANKING_FUNC_APP_NAME') or f'func-private-banking-mcp-{_mcp_suffix}'
FUNC_BASE_URL = f'https://{FUNC_APP_NAME}.azurewebsites.net'

# Re-fetch the system key (do not store it in .env)
MCP_KEY = subprocess.run(
    f'az functionapp keys list -g "{PB_MCP_RG}" -n "{FUNC_APP_NAME}" '
    f'--query "systemKeys.mcp_extension" -o tsv',
    shell=True, capture_output=True, text=True).stdout.strip()
MCP_SSE_URL = f'{FUNC_BASE_URL}/runtime/webhooks/mcp/sse?code={MCP_KEY}'

AGENT_NAME = 'aria-rm-briefing-agent'

print(f'Project endpoint : {PROJECT_ENDPOINT}')
print(f'Function app     : {FUNC_APP_NAME}')
print(f'Agent name       : {AGENT_NAME}')


Project endpoint : https://aif-core-c2676f.services.ai.azure.com/api/projects/project-admin-c2676f
Function app     : func-private-banking-mcp-97aab2
Agent name       : aria-rm-briefing-agent


## Authenticate and connect

In [2]:
from azure.ai.projects import AIProjectClient
from azure.identity import DefaultAzureCredential

project_client = AIProjectClient(endpoint=PROJECT_ENDPOINT, credential=DefaultAzureCredential())
openai_client  = project_client.get_openai_client()

# Always pull the latest version of the agent (versions are append-only on the new API).
versions = list(project_client.agents.list_versions(agent_name=AGENT_NAME))
agent = versions[0]
print(f"Connected to agent '{agent.name}' v{agent.version}")


Connected to agent 'aria-rm-briefing-agent' v1


Set up the `ask()` helper\n\nThe new-API invocation goes through the Responses surface — `openai_client.responses.create(..., extra_body={'agent_reference': ...})` — exactly the same pattern as [`08-08-human-in-the-loop/hitl.ipynb`](../08-08-human-in-the-loop/hitl.ipynb). MCP tool calls and their outputs appear inline in `response.output`.

In [3]:
AGENT_REF = {'agent_reference': {'name': agent.name, 'version': agent.version, 'type': 'agent_reference'}}


def ask(prompt: str, *, show_tool_calls: bool = True) -> str:
    """Send one prompt to the agent; return the final text response.

    `require_approval='never'` lives on the agent's tool definition (see Phase 5
    in 08-05b-01), so no per-run approval plumbing is needed here.
    """
    response = openai_client.responses.create(
        input=[{'role': 'user', 'content': prompt}],
        extra_body=AGENT_REF,
    )
    if show_tool_calls:
        for item in response.output:
            t = getattr(item, 'type', '')
            if 'call' in t.lower() or 'tool' in t.lower():
                name = getattr(item, 'name', None) or getattr(item, 'server_label', '') or t
                print(f'  ⇢ {t}: {name}')
    return response.output_text


---
## Part 1 — Intent-level workflows (one verb per intent)

Each query below is something an RM would actually say at 8:30am the morning of
a client meeting. Watch the tool calls: one intent → one tool.


### Demo 1.1 — Headline briefing

In [4]:
print(ask(
    "I have a 9am with the Berger family for their quarterly review. "
    "Give me the briefing — portfolio, any drift, recent activity, anything CRM has flagged, "
    "and the 2-3 things I should be ready to talk about."
))


  ⇢ mcp_list_tools: contoso_private_banking
  ⇢ mcp_call: cpb_prepare_client_briefing
  ⇢ mcp_call: cpb_run_query
  ⇢ mcp_call: cpb_prepare_client_briefing
Here is your briefing for the 9am quarterly review with the Berger Family Trust (UHNW Multi-Generation, RM Anna Müller):

Portfolio Summary as of 2026-05-10:
- AUM: CHF 89.53 million (base currency CHF)
- Asset allocation: Equities 51.51%, Fixed Income 32.37%, Alternatives 11.09%, Cash 5.03%
- Regional allocation: Switzerland 48.68%, North America 17.31%, Europe ex CH 22.78%, Global 11.24%
- ESG weighted score: 7.58 (ESG mandate in place)
- Top 5 positions:
  1. Contoso Swiss Equity Core (Equities) CHF 20.57m (22.97% of AUM)
  2. Contoso CHF Government Bond Long Duration (Fixed Income) CHF 18.52m (20.68%)
  3. Contoso US Large-Cap Quality (Equities) CHF 12.04m (13.45%)
  4. Contoso EUR Investment Grade Credit (Fixed Income) CHF 10.46m (11.69%)
  5. Contoso Climate Solutions Equity (Equities) CHF 10.06m (11.24%)

Portfolio Drift vs I

### Demo 1.2 — Drift-only deep dive

In [5]:
print(ask(
    "Show me the Lindemann family office's portfolio drift over 5 percentage points. "
    "I want to know which buckets are out of band before I look at rebalancing."
))


  ⇢ mcp_list_tools: contoso_private_banking
  ⇢ mcp_call: cpb_analyze_portfolio_drift
  ⇢ mcp_call: cpb_run_query
  ⇢ mcp_call: cpb_analyze_portfolio_drift
The Lindemann Family Office portfolio shows the following drifts over 5 percentage points:

Asset Class Drift:
- Fixed Income: Target 20.0%, Actual 13.25%, Drift -6.75% (over band)
- Alternatives: Target 25.0%, Actual 31.25%, Drift +6.25% (over band)

Regional Drift:
- Global: Target 0.0%, Actual 24.23%, Drift +24.23% (material breach)
- Europe ex-CH: Target 30.0%, Actual 20.27%, Drift -9.73% (material breach)
- Switzerland: Target 10.0%, Actual 3.77%, Drift -6.23% (over band)

There are no concentration flags or ESG floor breaches noted at this threshold.

Would you like me to prepare a full briefing for the upcoming meeting or find relevant research explaining the main drivers for these drifts? 

Citations: ips/cli-003, portfolio/cli-003


### Demo 1.3 — Research lookup against client holdings

In [6]:
print(ask(
    "Anything been written recently about AI infrastructure capex that would be relevant "
    "to Müller? She owns the Contoso AI & Infrastructure Thematic and asked at the last "
    "meeting whether we still believe in the cycle."
))


  ⇢ mcp_list_tools: contoso_private_banking
  ⇢ mcp_call: cpb_find_relevant_research
  ⇢ mcp_call: cpb_run_query
  ⇢ mcp_call: cpb_find_relevant_research
Regarding AI infrastructure capex relevant to Ms. Müller, who holds the Contoso AI & Infrastructure Thematic, there are recent pieces worth noting:

1. Research titled "AI infrastructure capex cycle — implications for Contoso AI & Infrastructure Thematic" (published 2026-04-30) highlights the 2026-2028 hyperscaler AI capex cycle supporting the thematic. It identifies three leading sub-themes: accelerator silicon and packaging, datacenter power and cooling, and grid-edge build-out. The fund is overweight power and cooling and underweight pure-play model providers due to valuation considerations. [research/res-003]

2. Market commentary "AI capex digestion debate — what we are watching" (published 2026-04-28) outlines three signals for the AI infrastructure capex debate: hyperscaler 2026-Q2 earnings commentary on 2027 capex guidance, up

### Demo 1.4 — Recent activity

In [7]:
print(ask(
    "Summarise what's been happening on the Riedi pension over the last 90 days. "
    "Group it by intent so I can explain it to the trustee chair."
))


  ⇢ mcp_list_tools: contoso_private_banking
  ⇢ mcp_call: cpb_summarize_recent_activity
I could not find a client with the ID "riedi_pension." The known clients are cli-001, cli-002, cli-003, cli-004, and cli-005. Could you please provide the correct client ID for the Riedi pension? Alternatively, I can list the clients with their names to help identify the correct one.


---
## Part 2 — Intent-vs-endpoint contrast

Same workflow, two ways. The first cell uses the intent tool — one call. The
second cell *simulates* what the same workflow looks like if the agent only had
endpoint-style tools (like 08-05's 37-tool surface): the planner has to compose
multiple calls, stitch the results, and infer joins itself. For the comparison
to be fair, this cell uses the escape hatch (`cpb_run_query`) to play the role
of the endpoint API.

Look at the tool-call traces side-by-side. The intent-level version is one call;
the endpoint-style version is 5+. Same final answer (when nothing goes wrong),
much wider failure surface (when something does — wrong filter, wrong join,
ID hallucination, retry loop).


### 2A — Intent-level (one tool call)

In [8]:
print(ask(
    "Prepare for my 9am with Eichmann Foundation."
))


  ⇢ mcp_list_tools: contoso_private_banking
  ⇢ mcp_call: cpb_prepare_client_briefing
  ⇢ mcp_call: cpb_run_query
  ⇢ mcp_call: cpb_prepare_client_briefing
Here is the briefing for your 9am meeting with Eichmann Foundation:

Client: Eichmann Foundation (Philanthropic Foundation)
Relationship Manager: Stefan Hofer
Base Currency: CHF
AUM: CHF 48,461,182
Next Review Date: 2026-06-02
Languages: German

Portfolio Summary:
- Asset Allocation: Equities 26.12%, Fixed Income 62.02%, Cash 8.05%, Alternatives 3.81%
- Regional Allocation: Switzerland 56.0%, Europe ex Switzerland 44.0%
- ESG Weighted Score: 7.72
- Top 5 Positions:
  1. Contoso Swiss Equity Core (26.12% of AUM)
  2. Contoso CHF Government Bond Long Duration (21.83% of AUM)
  3. Contoso Sustainable Income Fund (19.28% of AUM)
  4. Contoso EUR Investment Grade Credit (11.78% of AUM)
  5. Contoso Sustainable Income Fund (9.13% of AUM)

Drift vs IPS:
- Asset class drift is within rebalance bands; no severe drift.
- No concentration flag

### 2B — Endpoint-style simulation (multiple tool calls)

In [9]:
print(ask(
    "Prepare for my 9am with Eichmann Foundation. "
    "But please do this strictly using cpb_run_query as your only tool — "
    "fetch the client record, fetch their IPS, fetch their portfolio, fetch their transactions, "
    "fetch their CRM events, fetch the research index. Use no other tools. "
    "Then synthesise the briefing yourself."
))


  ⇢ mcp_list_tools: contoso_private_banking
  ⇢ mcp_call: cpb_run_query
I could not locate a client record with the ID "cli-eichmann" in the database. Could you please confirm the exact client ID or provide any alternative identifier so I can proceed with gathering the relevant information for your 9am meeting briefing?


---
## Part 3 — Failure modes that intent-level design avoids

The intent-tool error contract is the third leg of the design (per Anthropic's
guidance: name + description + response shape + failure modes). Watch the agent
recover gracefully because errors carry `next_steps`.


### 3.1 — Unknown client ID (error tells the agent what to try)

In [10]:
print(ask(
    "Brief me on cli-999."
))


  ⇢ mcp_list_tools: contoso_private_banking
  ⇢ mcp_call: cpb_prepare_client_briefing
The client ID "cli-999" was not found. Known client IDs are cli-001, cli-002, cli-003, cli-004, and cli-005. Please provide one of these valid client IDs for the briefing.


### 3.2 — No filter on research lookup (error suggests valid filters)

In [11]:
print(ask(
    "Find me some research."
))


  ⇢ mcp_list_tools: contoso_private_banking
Could you please specify the topic, client, or instrument (ISIN) you want the research on? This will help me find the most relevant research for you.


---
## Notes

- The intent-level design choice is what makes the same agent useful for an RM,
  an analyst, and a compliance reviewer without proliferating tools.
- Compare the tool-call traces in Part 2 with the queries in
  [`08-05-02-contoso-pmo-agent-queries.ipynb`](../08-05-contoso-pmo-mcp/08-05-02-contoso-pmo-agent-queries.ipynb)
  for the cleanest illustration of the contrast.
- Workflow-level evals (the [`tests/workflow_evals.jsonl`](tests/workflow_evals.jsonl)
  file) catch the failure mode where renaming a tool or rewording a description
  silently degrades selection accuracy. Re-run them whenever the function-app
  registry changes.
